# 10장 웹부록 | 격자 지도에서 에이스타로 길 찾기

「영화로 보는 AI 리터러시」 10장 '운전대 없는 자동차: 자율주행과 드론'의 코랩 실습입니다.

**쓰는 법**

- 코드를 몰라도 괜찮습니다. 위에서부터 차례로 셀 왼쪽의 ▶(실행) 버튼을 누르세요.
- `# ← 이 숫자를 바꿔 보세요` 표시가 있는 곳만 고치고, 그 셀부터 다시 실행해 보세요.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 따로 설치할 것은 없습니다. 그래프 글자는 영어로 나오고, 설명은 이 글에 우리말로 적었습니다.

**이 실습에서 할 일**

1. 10×10 격자 지도에 장애물을 놓습니다.
2. 다익스트라와 에이스타 알고리즘이 길을 찾는 과정을 한 칸씩 움직이는 그림으로 봅니다.
3. 두 알고리즘이 살펴본 칸의 수와 찾은 길의 길이를 표로 비교합니다.
4. 끝으로 새 떼 규칙(보이즈) 세 가지의 세기를 바꿔 가며 가상 드론 50대의 움직임을 관찰합니다.

책 3-6절의 '심화 상식 | 경로 계획 알고리즘'과 3-8절 '하늘의 자율주행', 실습 2 '몸으로 하는 군집 비행'을 함께 보세요. 전체 실행에 30초쯤 걸립니다.

## 1단계. 격자 지도 만들기

지도를 가로 10칸, 세로 10칸으로 나눕니다. 자동차는 한 번에 위·아래·왼쪽·오른쪽으로 한 칸씩만 갈 수 있고, 한 칸을 갈 때마다 비용이 1씩 듭니다.

- `S`는 출발 칸, `G`는 도착 칸입니다.
- `#`은 장애물(건물, 공사 구간)입니다. 지나갈 수 없습니다.

👉 **바꿔 볼 곳**: 아래 지도 글자를 직접 고쳐 보세요. `.`을 `#`으로 바꾸면 장애물이 생깁니다. 줄마다 글자 수는 10개를 지켜 주세요.

In [ ]:
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
plt.rcParams["animation.embed_limit"] = 40   # 움직이는 그림 크기 한도(MB)

MAP_TEXT = """
S.........
..........
..#######.
........#.
........#.
....#...#.
....#...#.
....#####.
..........
.........G
"""   # ← 이 지도를 고쳐 보세요 (. 길, # 장애물, S 출발, G 도착)

def read_map(text):
    rows = [r for r in text.strip().splitlines()]
    grid = np.array([[c == "#" for c in r] for r in rows])
    start = goal = None
    for y, r in enumerate(rows):
        for x, c in enumerate(r):
            if c == "S": start = (y, x)
            if c == "G": goal = (y, x)
    return grid, start, goal

grid, start, goal = read_map(MAP_TEXT)

def show_map(ax, grid, start, goal, title=""):
    ax.imshow(grid, cmap="Greys", vmin=0, vmax=1.3)
    ax.text(start[1], start[0], "S", ha="center", va="center", color="green", fontsize=14, weight="bold")
    ax.text(goal[1], goal[0], "G", ha="center", va="center", color="blue", fontsize=14, weight="bold")
    ax.set_xticks(np.arange(-.5, grid.shape[1]), minor=True); ax.set_yticks(np.arange(-.5, grid.shape[0]), minor=True)
    ax.grid(which="minor", color="lightgray"); ax.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(4, 4)); show_map(ax, grid, start, goal, "10 x 10 map"); plt.show()
print("출발:", start, " 도착:", goal, " 장애물 칸 수:", grid.sum())

## 2단계. 두 알고리즘 만들기

책에 나온 두 탐험가를 코드로 옮깁니다.

- **다익스트라**(1959): 출발점에서 **가까운 칸부터** 차례로 '여기까지 오는 가장 짧은 거리'를 적어 나갑니다. 물감이 사방으로 번지듯 찾습니다.
- **에이스타**(1968): '지금까지 온 거리 + 도착까지 남은 거리의 **어림값**'이 가장 작은 칸부터 넓혀 갑니다. 어림값은 장애물을 무시하고 가로로 몇 칸, 세로로 몇 칸 남았는지 더한 값(맨해튼 거리)입니다. 깃발을 보며 그쪽으로 먼저 칠해 가는 탐험가입니다.

두 알고리즘은 코드가 거의 같습니다. 다른 곳은 단 한 줄, '어림값을 더하느냐'뿐입니다. 아래 코드에서 `use_guess` 부분을 찾아보세요.

In [ ]:
def find_path(grid, start, goal, use_guess):
    """use_guess=False면 다익스트라, True면 에이스타입니다.
    돌려주는 것: 찾은 길, 살펴본 칸의 순서"""
    H, W = grid.shape
    def guess(cell):                       # 도착까지 남은 거리의 어림값(맨해튼 거리)
        return abs(cell[0] - goal[0]) + abs(cell[1] - goal[1])
    best = {start: 0}                      # 지금까지 알아낸 '여기까지 가장 짧은 거리'
    came_from = {}
    queue = [(0, 0, start)]                # (우선순위, 어림값, 칸)
    visited_order, done = [], set()
    while queue:
        _, _, cell = heapq.heappop(queue)
        if cell in done:
            continue
        done.add(cell); visited_order.append(cell)
        if cell == goal:
            break
        y, x = cell
        for nxt in [(y-1, x), (y+1, x), (y, x-1), (y, x+1)]:
            ny, nx = nxt
            if 0 <= ny < H and 0 <= nx < W and not grid[ny, nx]:
                cost = best[cell] + 1
                if cost < best.get(nxt, 1e9):
                    best[nxt] = cost; came_from[nxt] = cell
                    h = guess(nxt)
                    priority = cost + h if use_guess else cost   # ← 두 알고리즘의 유일한 차이
                    heapq.heappush(queue, (priority, h, nxt))
    if goal not in came_from and goal != start:
        return None, visited_order
    path = [goal]
    while path[-1] != start:
        path.append(came_from[path[-1]])
    return path[::-1], visited_order

results = {}
for name, g in [("Dijkstra", False), ("A*", True)]:
    path, visited = find_path(grid, start, goal, use_guess=g)
    results[name] = (path, visited)
    if path is None:
        print(f"{name}: 길이 없습니다! 지도를 다시 고쳐 주세요.")
    else:
        print(f"{name}: 길 {len(path)-1}칸, 살펴본 칸 {len(visited)}개")

## 3단계. 한 칸씩 움직이는 그림으로 비교하기

아래 그림의 ▶ 버튼을 누르면 두 탐험가가 동시에 칸을 살펴 나갑니다. 색칠된 칸이 '살펴본 칸'이고, 마지막에 빨간 선이 찾은 길입니다. 속도 조절과 한 칸씩 넘기기도 됩니다.

그림을 만드는 데 10초쯤 걸립니다.

In [ ]:
def animate_search(results, grid, start, goal):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4.2))
    n_frames = max(len(v[1]) for v in results.values()) + 4
    layers = {}
    for ax, (name, (path, visited)) in zip(axes, results.items()):
        show_map(ax, grid, start, goal, name)
        img = ax.imshow(np.full(grid.shape, np.nan), cmap="YlOrRd", vmin=0, vmax=n_frames, alpha=0.7)
        line, = ax.plot([], [], color="red", lw=3)
        layers[name] = (img, line, visited, path, ax)
    def update(f):
        out = []
        for name, (img, line, visited, path, ax) in layers.items():
            shown = np.full(grid.shape, np.nan)
            for i, (y, x) in enumerate(visited[:f+1]):
                shown[y, x] = i
            img.set_data(shown)
            k = min(f + 1, len(visited))
            ax.set_title(f"{name}: {k} cells checked")
            if f >= len(visited) and path:
                line.set_data([p[1] for p in path], [p[0] for p in path])
            out += [img, line]
        return out
    anim = animation.FuncAnimation(fig, update, frames=n_frames, interval=150)
    plt.close(fig)
    return anim

anim = animate_search(results, grid, start, goal)
HTML(anim.to_jshtml())

움직이는 그림이 보이지 않으면 아래 셀이 같은 내용을 멈춘 그림으로 보여 줍니다. 칸 안의 숫자는 '몇 번째로 살펴본 칸인지'입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (name, (path, visited)) in zip(axes, results.items()):
    show_map(ax, grid, start, goal, f"{name}: {len(visited)} cells checked")
    for i, (y, x) in enumerate(visited):
        ax.text(x, y, str(i + 1), ha="center", va="center", fontsize=7, color="darkorange")
    if path:
        ax.plot([p[1] for p in path], [p[0] for p in path], color="red", lw=3, alpha=0.6)
plt.tight_layout(); plt.show()

## 4단계. 표로 비교하기

두 알고리즘이 찾은 길의 길이는 같습니다. 둘 다 '가장 짧은 길'을 보장합니다. 차이는 그 길을 찾기 위해 **얼마나 많은 칸을 살펴봤느냐**입니다. 자율주행차는 이 계산을 1초에도 여러 번 해야 하므로, 덜 살펴보고도 같은 답을 찾는 것이 중요합니다.

지도가 커지면 차이도 커집니다. 아래에서는 같은 비교를 가로세로 30칸짜리 무작위 지도에서도 해 봅니다.

👉 **바꿔 볼 곳**: `BIG_SIZE`(지도 크기)와 `OBSTACLE_RATE`(장애물 비율)를 바꿔 보세요.

In [ ]:
rows = []
for name, (path, visited) in results.items():
    rows.append({"지도": "10x10 (위의 지도)", "알고리즘": name,
                 "살펴본 칸 수": len(visited), "찾은 길의 길이(칸)": len(path) - 1 if path else None})

BIG_SIZE = 30         # ← 이 숫자를 바꿔 보세요 (지도 한 변의 칸 수)
OBSTACLE_RATE = 0.25  # ← 이 숫자를 바꿔 보세요 (0.0 ~ 0.4, 장애물이 차지하는 비율)

rng = np.random.default_rng(10)
for attempt in range(50):          # 길이 있는 지도가 나올 때까지 다시 만듭니다
    big = rng.random((BIG_SIZE, BIG_SIZE)) < OBSTACLE_RATE
    s, g = (0, 0), (BIG_SIZE - 1, BIG_SIZE - 1)
    big[s] = big[g] = False
    big_results = {name: find_path(big, s, g, use_guess=u) for name, u in [("Dijkstra", False), ("A*", True)]}
    if big_results["A*"][0] is not None:
        break
for name, (path, visited) in big_results.items():
    rows.append({"지도": f"{BIG_SIZE}x{BIG_SIZE} 무작위", "알고리즘": name,
                 "살펴본 칸 수": len(visited), "찾은 길의 길이(칸)": len(path) - 1 if path else None})
display(pd.DataFrame(rows))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (name, (path, visited)) in zip(axes, big_results.items()):
    show_map(ax, big, s, g, f"{name}: {len(visited)} cells checked")
    seen = np.full(big.shape, np.nan)
    for i, (y, x) in enumerate(visited):
        seen[y, x] = i
    ax.imshow(seen, cmap="YlOrRd", alpha=0.7)
    if path:
        ax.plot([p[1] for p in path], [p[0] for p in path], color="red", lw=2)
plt.tight_layout(); plt.show()

## 5단계. 새 떼 규칙으로 드론 50대 날리기 (보이즈)

이번에는 하늘로 갑니다. 1987년 크레이그 레이놀즈가 보여 준 **보이즈**의 세 규칙을 가상 드론 50대에게 줍니다. 드론은 중앙의 명령 없이 **자기 주변(반경 안의 이웃)만** 봅니다.

1. **떨어지기**(separation): 이웃과 너무 가까우면 멀어집니다.
2. **줄 맞추기**(alignment): 이웃이 가는 방향으로 나도 갑니다.
3. **모이기**(cohesion): 이웃들의 가운데 쪽으로 다가갑니다.

하늘은 가로세로 20m짜리 정사각형이고, 한쪽 끝으로 나가면 반대쪽에서 다시 들어옵니다. 시뮬레이션이 끝나면 세 가지를 잽니다.

- **부딪힐 뻔한 횟수**: 두 드론이 0.3m보다 가까워진 횟수(책 실습 2의 표와 같은 항목)
- **방향 일치도**: 모든 드론이 같은 방향으로 날면 1, 제각각이면 0에 가깝습니다.
- **무리의 수**: 서로 이웃인 드론끼리 묶었을 때 몇 무리로 나뉘었는지

👉 **바꿔 볼 곳**: 세 규칙의 세기를 0(끔)부터 3(아주 셈)까지 바꿔 보세요.

In [ ]:
SEPARATION = 1.5   # ← 이 숫자를 바꿔 보세요 (떨어지기 세기)
ALIGNMENT  = 1.0   # ← 이 숫자를 바꿔 보세요 (줄 맞추기 세기)
COHESION   = 1.0   # ← 이 숫자를 바꿔 보세요 (모이기 세기)

N_DRONES, SKY, SEE, TOO_CLOSE, SPEED = 50, 20.0, 3.0, 1.0, 1.0

def wrap(d):                      # 하늘 끝과 끝이 이어져 있으므로 가장 가까운 방향으로 잽니다
    return (d + SKY/2) % SKY - SKY/2

def fly(sep, ali, coh, steps=300, seed=0, leader=None, keep=False):
    rng = np.random.default_rng(seed)
    pos = rng.uniform(0, SKY, (N_DRONES, 2))
    ang = rng.uniform(0, 2*np.pi, N_DRONES)
    vel = np.c_[np.cos(ang), np.sin(ang)] * SPEED
    near_miss, frames = 0, []
    for t in range(steps):
        d = wrap(pos[None, :, :] - pos[:, None, :])        # 드론 i에서 j로 가는 화살표
        dist = np.linalg.norm(d, axis=2) + np.eye(N_DRONES) * 1e9
        nb = dist < SEE                                     # 내가 볼 수 있는 이웃
        cnt = np.maximum(nb.sum(1, keepdims=True), 1)
        close = dist < TOO_CLOSE
        f_sep = -(d * (close / dist**2)[:, :, None]).sum(1)                  # 1) 떨어지기
        f_ali = (nb[:, :, None] * vel[None, :, :]).sum(1) / cnt - vel        # 2) 줄 맞추기
        f_coh = (nb[:, :, None] * d).sum(1) / cnt                            # 3) 모이기
        f_coh[nb.sum(1) == 0] = 0
        steer = sep * f_sep + ali * f_ali + coh * 0.3 * f_coh + rng.normal(0, 0.1, (N_DRONES, 2))
        if leader is not None:                              # 몰래 길잡이 1대 (6단계)
            steer[0] = 2.0 * (np.array(leader) / np.linalg.norm(leader) * SPEED - vel[0])
        vel = vel + 0.1 * steer
        vel = vel / np.linalg.norm(vel, axis=1, keepdims=True) * SPEED   # 속도는 일정하게
        pos = (pos + 0.1 * vel) % SKY
        near_miss += int(np.triu(dist < 0.3).sum())
        if keep and t % 5 == 0:
            frames.append((pos.copy(), vel.copy()))
    order = np.linalg.norm(vel.mean(0)) / SPEED
    groups = count_groups(pos)
    return dict(near_miss=near_miss, order=order, groups=groups, pos=pos, vel=vel, frames=frames)

def count_groups(pos):
    d = np.linalg.norm(wrap(pos[None] - pos[:, None]), axis=2)
    link = d < SEE
    seen, groups = set(), 0
    for i in range(len(pos)):
        if i in seen: continue
        groups += 1; stack = [i]
        while stack:
            k = stack.pop()
            if k in seen: continue
            seen.add(k); stack += list(np.where(link[k])[0])
    return groups

def draw_sky(ax, pos, vel, title):
    ax.quiver(pos[:, 0], pos[:, 1], vel[:, 0], vel[:, 1], color="tab:blue", scale=25, width=0.006)
    ax.set_xlim(0, SKY); ax.set_ylim(0, SKY); ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

mine = fly(SEPARATION, ALIGNMENT, COHESION, keep=True)
print(f"내 설정 → 부딪힐 뻔한 횟수 {mine['near_miss']}, 방향 일치도 {mine['order']:.2f}, 무리 {mine['groups']}개")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, k, label in zip(axes, [0, len(mine["frames"])//4, -1], ["start", "a little later", "end"]):
    p, v = mine["frames"][k]
    draw_sky(ax, p, v, label)
plt.tight_layout(); plt.show()

아래 셀은 위 드론 무리의 움직임을 움직이는 그림으로 보여 줍니다. 화살표 하나가 드론 한 대이고, 화살표 방향이 나는 방향입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
def upd(k):
    ax.clear(); p, v = mine["frames"][k]
    draw_sky(ax, p, v, f"Boids: step {k*5}")
anim2 = animation.FuncAnimation(fig, upd, frames=len(mine["frames"]), interval=80)
plt.close(fig)
HTML(anim2.to_jshtml())

### 규칙 하나씩 꺼 보기

책 실습 2의 표처럼 조건을 바꿔 비교합니다. 세 규칙을 모두 켠 경우와, 하나씩 끈 경우입니다.

In [ ]:
settings = [("all three rules", SEPARATION, ALIGNMENT, COHESION),
            ("no rules", 0, 0, 0),
            ("no separation", 0, ALIGNMENT, COHESION),
            ("no alignment", SEPARATION, 0, COHESION),
            ("no cohesion", SEPARATION, ALIGNMENT, 0)]
names_ko = {"all three rules": "세 규칙 모두", "no rules": "규칙 없음", "no separation": "떨어지기 끔",
            "no alignment": "줄 맞추기 끔", "no cohesion": "모이기 끔"}
rows, outs = [], []
for label, s_, a_, c_ in settings:
    r = fly(s_, a_, c_)
    outs.append((label, r))
    rows.append({"조건": names_ko[label], "부딪힐 뻔한 횟수": r["near_miss"],
                 "방향 일치도(0~1)": round(r["order"], 2), "무리의 수": r["groups"]})
display(pd.DataFrame(rows))

fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, (label, r) in zip(axes, outs):
    draw_sky(ax, r["pos"], r["vel"], label)
plt.tight_layout(); plt.show()

**읽는 법**

- **규칙 없음**: 드론이 제각각 날아 방향 일치도가 낮습니다.
- **떨어지기 끔**: 무리는 생기지만 서로 너무 붙어 '부딪힐 뻔한 횟수'가 크게 늘어납니다.
- **줄 맞추기 끔**: 가까이 모여도 저마다 다른 방향으로 납니다.
- **모이기 끔**: 같은 방향으로 날지만 작은 무리로 흩어지기 쉽습니다.

아무도 '이쪽으로 가자'고 명령하지 않았는데 무리가 한 방향으로 움직입니다. 책에서 말한 **창발**입니다.

## 6단계. 더 해 보기: 몰래 길잡이 한 대

책 실습 2의 3회차처럼, 드론 한 대에게만 몰래 '오른쪽 위로 가라'는 명령을 줍니다. 나머지 49대는 여전히 이웃만 봅니다. 무리 전체가 따라가는지 확인해 보세요.

In [ ]:
LEADER_DIRECTION = (1.0, 1.0)   # ← 이 숫자를 바꿔 보세요 (길잡이가 갈 방향: 가로, 세로)

r_lead = fly(SEPARATION, ALIGNMENT, COHESION, leader=LEADER_DIRECTION)
v = r_lead["vel"]
target = np.array(LEADER_DIRECTION) / np.linalg.norm(LEADER_DIRECTION)
follow = (v[1:] @ target / SPEED > 0.7).mean()
print(f"길잡이와 비슷한 방향(차이 45도 안쪽)으로 나는 드론: 49대 중 {follow*49:.0f}대")
fig, ax = plt.subplots(figsize=(4.5, 4.5))
draw_sky(ax, r_lead["pos"], r_lead["vel"], "One secret leader (red)")
ax.quiver(*r_lead["pos"][0], *r_lead["vel"][0], color="red", scale=25, width=0.012)
plt.show()

## 더 해 보기: 에이스타를 속이는 함정 지도

1단계의 `MAP_TEXT`를 아래 지도로 바꾸고 모두 다시 실행해 보세요. 도착점 쪽이 벽으로 막혀 있어, '깃발만 보고 가는' 에이스타가 벽 앞 주머니를 한참 뒤지게 됩니다. 그래도 다익스트라보다 덜 살펴볼까요?

```
S.......#.
........#.
........#.
........#.
........#.
........#.
........#.
.########.
..........
.........G
```

## 생각해 볼 것

1. 에이스타는 '남은 거리의 어림값' 덕분에 덜 살펴보고도 같은 길을 찾았습니다. 그런데 어림값이 실제보다 **크게** 틀린다면(예: 남은 거리를 두 배로 부풀리면) 어떤 일이 생길까요? 2단계 코드의 `cost + h`를 `cost + 2*h`로 바꿔 실험해 보고, 자율주행차에서 '빠른 계산'과 '가장 좋은 길' 중 무엇을 앞세워야 할지 생각해 보세요.
2. 지도에 없는 공사 구간이 갑자기 나타나면 미리 찾은 길은 쓸모가 없어집니다. 책 3-1절의 '인지-판단-제어'를 되풀이하는 순환과 연결해, 자율주행차가 길 찾기를 왜 계속 다시 해야 하는지 설명해 보세요.
3. 6단계에서 길잡이 한 대가 무리 전체를 끌고 갔다면, **해킹된 드론 한 대**는 군집 전체에 어떤 영향을 줄 수 있을까요? 중앙 컴퓨터가 모두 정해 주는 드론 쇼 방식과 비교해 장단점을 쓰세요.